# Liste înlănțuite

O **listă înlănțuită** este o secvență de noduri, fiecare nod reținând o valoare și un pointer (referință) către nodul următor.

| Operație | Vector (array) | Listă simplu înlănțuită |
|---|---|---|
| Acces la poziția `i` | O(1) | O(n) |
| Inserare la început | O(n) | O(1) |
| Inserare la final | O(1) amortizat | O(1) (dacă reținem `tail`) |
| Ștergere nod (având pointer la predecesor) | O(n) | O(1) |
| Căutare | O(n) | O(n) |

Avantaj: inserări/ștergeri ieftine, fără mutarea elementelor. Dezavantaj: fără acces direct, memorie în plus pentru pointeri.

## 1. Listă simplu înlănțuită

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next


class SinglyLinkedList:
    def __init__(self):
        self.head = None   # primul nod
        self.tail = None   # ultimul nod (pentru inserare la final în O(1))
        self.size = 0

    def push_front(self, value):
        node = Node(value, self.head)
        self.head = node
        if self.tail is None:
            self.tail = node
        self.size += 1

    def push_back(self, value):
        node = Node(value)
        if self.tail is None:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self.size += 1

    def insert_after(self, node, value):
        """Inserează după un nod dat - O(1)."""
        new_node = Node(value, node.next)
        node.next = new_node
        if node is self.tail:
            self.tail = new_node
        self.size += 1

    def find(self, value):
        current = self.head
        while current is not None:
            if current.value == value:
                return current
            current = current.next
        return None

    def remove(self, value):
        """Șterge prima apariție a valorii. Întoarce True dacă a găsit-o."""
        prev, current = None, self.head
        while current is not None and current.value != value:
            prev, current = current, current.next
        if current is None:
            return False
        if prev is None:              # ștergem capul
            self.head = current.next
        else:
            prev.next = current.next
        if current is self.tail:      # ștergem ultimul nod
            self.tail = prev
        self.size -= 1
        return True

    def reverse(self):
        prev, current = None, self.head
        self.tail = self.head
        while current is not None:
            nxt = current.next
            current.next = prev
            prev, current = current, nxt
        self.head = prev

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.value
            current = current.next

    def __len__(self):
        return self.size

    def __repr__(self):
        return " -> ".join(map(str, self)) + " -> None"

In [ ]:
lst = SinglyLinkedList()
for x in [3, 7, 1]:
    lst.push_back(x)
lst.push_front(10)
print(lst, "| lungime:", len(lst))

lst.insert_after(lst.find(7), 99)
print("după inserare după 7:", lst)

lst.remove(10)
lst.remove(1)
print("după ștergeri:      ", lst)

lst.reverse()
print("inversată:          ", lst)

## 2. Listă dublu înlănțuită

Fiecare nod reține și un pointer `prev`. Putem parcurge în ambele sensuri și putem șterge un nod în O(1) având doar pointer la el.

In [ ]:
class DNode:
    def __init__(self, value):
        self.value = value
        self.prev = None
        self.next = None


class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None

    def push_front(self, value):
        node = DNode(value)
        node.next = self.head
        if self.head is not None:
            self.head.prev = node
        else:
            self.tail = node
        self.head = node
        return node

    def push_back(self, value):
        node = DNode(value)
        node.prev = self.tail
        if self.tail is not None:
            self.tail.next = node
        else:
            self.head = node
        self.tail = node
        return node

    def remove_node(self, node):
        """Ștergere în O(1), având referința la nod."""
        if node.prev is not None:
            node.prev.next = node.next
        else:
            self.head = node.next
        if node.next is not None:
            node.next.prev = node.prev
        else:
            self.tail = node.prev

    def forward(self):
        current, result = self.head, []
        while current:
            result.append(current.value)
            current = current.next
        return result

    def backward(self):
        current, result = self.tail, []
        while current:
            result.append(current.value)
            current = current.prev
        return result

In [ ]:
dll = DoublyLinkedList()
dll.push_back(2)
middle = dll.push_back(5)
dll.push_back(8)
dll.push_front(1)
print("înainte:", dll.forward())
print("înapoi: ", dll.backward())

dll.remove_node(middle)
print("după ștergerea lui 5:", dll.forward())

## 3. Probleme clasice

- **Mijlocul listei**: doi pointeri, unul avansează câte 1, celălalt câte 2.
- **Detectarea unui ciclu** (algoritmul lui Floyd, „iepurele și broasca țestoasă”): dacă pointerul rapid îl prinde pe cel lent, lista are ciclu.

In [ ]:
def find_middle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
    return slow


def has_cycle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False


lst = SinglyLinkedList()
for x in range(1, 8):
    lst.push_back(x)
print(lst)
print("mijloc:", find_middle(lst.head).value)
print("are ciclu?", has_cycle(lst.head))

lst.tail.next = lst.head.next.next   # facem un ciclu 7 -> 3
print("după ce legăm 7 -> 3, are ciclu?", has_cycle(lst.head))